# Overview

This notebook id dedicated for evaluating NER model on a benchmark dataset.

# Step 0 - Setup
Run the code below to mount your Google Drive and most of the necessary packages to carry out the evaluation

**Action:**
No code changes required. When prompted, connect your Google account

In [1]:
%%capture
!cd /content/
!rm -rf ./CASM_utils/
!pip install git+https://github.com/ay94/multilingual-ner.git!pip install -e CASM_utils/
!cd /content/CASM_utils


import CASM_utils
import importlib
from CASM_utils import utils
importlib.reload(utils)

In [2]:
## Mount GDrive
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

## Imports
import os
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from collections import Counter, defaultdict

Mounted at /content/drive/


In [4]:
# direct the file handler to the data folder
FOLDER = '/content/drive/MyDrive/CASM/FAST/Thai/NER/Benchmark'
fh = utils.FileHandler(FOLDER)

 # Read results
 - Read corss model results
 - Read corss datasets results
 -

wikineural

In [11]:
def extract_results():
  TAGs = ['LOC', 'PER', 'ORG']
  for TAG in TAGs:
    entity_results = []
    directory = Path(fh.cr_fn('outputs'))
    for subdir, dirs, files in os.walk(directory):
        for file in files:
            # Construct the file's full path
            file_path = os.path.join(subdir, file)
            excluded_folder = file_path.split('/')[-2]
            if excluded_folder != 'wikiann-span':
              # Open and read the file
              file_name = file_path.split('/')[-1]
              model_name = file_path.split('/')[-2]
              if 'seqeval' in file_name:
                df = pd.read_csv(file_path)
                df['Model Name'] = model_name
                df['Data Name'] = file_name.split('-')[0]
                print()
                display(df[df['Tag'] == TAG])
                entity_results.append(df[df['Tag'] == TAG])
    pd.concat(entity_results).to_csv(
        fh.cr_fn(f'outputs/consolidated-{TAG}.csv'),
        index=False
    )

In [12]:
extract_results()

,Tag,Precision,Recall,F1,support,Model Name,Data Name
0,LOC,0.8009,0.8391,0.8196,839,napatswift-xlm-thainer,thainer


,Tag,Precision,Recall,F1,support,Model Name,Data Name
0,LOC,0.8102,0.8749,0.8413,839,pythainlp-thainer-corpus,thainer


,Tag,Precision,Recall,F1,support,Model Name,Data Name
0,LOC,0.8528,0.8975,0.8746,839,Pavarissy-phayathaibert-thainer,thainer


,Tag,Precision,Recall,F1,support,Model Name,Data Name
2,PER,0.8885,0.9023,0.8954,645,napatswift-xlm-thainer,thainer


,Tag,Precision,Recall,F1,support,Model Name,Data Name
2,PER,0.8634,0.8915,0.8772,645,pythainlp-thainer-corpus,thainer


,Tag,Precision,Recall,F1,support,Model Name,Data Name
2,PER,0.9138,0.9364,0.925,645,Pavarissy-phayathaibert-thainer,thainer


,Tag,Precision,Recall,F1,support,Model Name,Data Name
1,ORG,0.8421,0.9089,0.8743,1285,napatswift-xlm-thainer,thainer


,Tag,Precision,Recall,F1,support,Model Name,Data Name
1,ORG,0.8504,0.8981,0.8736,1285,pythainlp-thainer-corpus,thainer


,Tag,Precision,Recall,F1,support,Model Name,Data Name
1,ORG,0.8753,0.9175,0.8959,1285,Pavarissy-phayathaibert-thainer,thainer


wikiann

In [13]:
LOC = pd.read_csv(
    fh.cr_fn(f'outputs/consolidated-LOC.csv')
)
PER = pd.read_csv(
    fh.cr_fn(f'outputs/consolidated-PER.csv')
)
ORG = pd.read_csv(
    fh.cr_fn(f'outputs/consolidated-ORG.csv')
)

In [14]:
LOC

,Tag,Precision,Recall,F1,support,Model Name,Data Name
0,LOC,0.8009,0.8391,0.8196,839,napatswift-xlm-thainer,thainer
1,LOC,0.8102,0.8749,0.8413,839,pythainlp-thainer-corpus,thainer
2,LOC,0.8528,0.8975,0.8746,839,Pavarissy-phayathaibert-thainer,thainer


In [8]:
f1_loc = LOC.groupby('Model Name')['F1'].mean()
f1_per = PER.groupby('Model Name')['F1'].mean()
f1_org = ORG.groupby('Model Name')['F1'].mean()

results = pd.DataFrame({
    'LOC': f1_loc,
    'PER': f1_per,
    'ORG': f1_org
}).reset_index().rename(columns={'Model Name': 'Model / Data Name'})

In [9]:
results

,Model / Data Name,LOC,PER,ORG
0,Pavarissy-phayathaibert-thainer,0.8746,0.9250,0.8959
1,napatswift-xlm-thainer,0.8196,0.8954,0.8743
2,pythainlp-thainer-corpus,0.8413,0.8772,0.8736


In [10]:
results.to_csv(
    fh.cr_fn(f'outputs/consolidated_average_entity.csv')

)